In [2]:
import pandas as pd

fname_base = 'D:\\PhD\\Openalex\\subset_ok\\years\\2019-2022\\'

INSTITUTIONS

In [8]:
# categorical attributes OHE, leave text for SentenceBert

I = pd.read_csv(fname_base+'institutions.csv') 
#I.columns --> Index(['id', 'name', 'ror', 'country-code', 'type'], dtype='object')

#verify nan 
cc = I['country-code'].drop_duplicates().tolist()
print(len(cc))
t = I['type'].drop_duplicates().tolist()
print(t)


97
['education', 'company', 'facility', 'other', 'healthcare', 'nonprofit', 'government', 'archive']


In [3]:
I[I['type'].isna()]



,id,name,ror,country-code,type


In [11]:
I['type'] = I['type'].fillna('education')
I.to_csv(fname_base+'institutions.csv', index=False)


In [28]:
#I['country-code'].isna().sum()
I['type'].isna().sum()

0

In [4]:
I.columns

Index(['id', 'name', 'ror', 'country-code', 'type'], dtype='object')

In [7]:
I.drop(columns=['ror','name'], axis=1, inplace=True)
I['type'] = I['type'].astype('category')
I['country-code'] = I['country-code'].astype('category')
I_ohe = pd.get_dummies(data=I, columns=['type', 'country-code'])
I_ohe.to_csv(fname_base+'attributes\\institutions_ohe.csv', index=False)
print(I_ohe.shape)

(1831, 106)


In [10]:
I_text = I[['id', 'name']]
I_text.to_csv(fname_base+'attributes\\institutions_text.csv', index=False)
I_text.shape

(1831, 2)

AUTHORS

In [2]:
A = pd.read_csv(fname_base+'authors_metadata.csv') 
A.columns

Index(['id', 'orcid', 'twitter_id', 'wikipedia_id', 'name', 'n_works', 'n_cit',
       'n_cit_year', 'impact_factor', 'h_index', 'i10_index',
       'last_institution_id', 'x_concepts'],
      dtype='object')

In [3]:
#n_works_by_year, n_cit_by_year, 

def encoding_n_works_cit(row, max_year): #data_year --> year, works_count, cited_by_count
    n_works = row['n_works']
    n_cit = row['n_cit']
    data_years = row['n_cit_year'][2:-2].split('}, {')
    for data_y in data_years:
        d = data_y.split(', ')
        year = int(d[0].split(': ')[1])
        if year > max_year:
            n_works -= int(d[1].split(': ')[1])
            n_cit -= int(d[2].split(': ')[1])
    row['n_works'] = n_works
    row['n_cit'] = n_cit
    return row

In [15]:
#TEST

row = A.head(1)

n_works = row['n_works']
print('N works', n_works)
n_cit = row['n_cit']
print('N cit', n_cit)
data_years = row['n_cit_year'].tolist()[0]
data_years = data_years[2:-2]
data_y = data_years.split('}, {')
for d in data_y:
    print(d)

N works 0    282
Name: n_works, dtype: int64
N cit 0    17698
Name: n_cit, dtype: int64
'year': 2023, 'works_count': 0, 'cited_by_count': 225
'year': 2022, 'works_count': 23, 'cited_by_count': 443
'year': 2021, 'works_count': 5, 'cited_by_count': 640
'year': 2020, 'works_count': 3, 'cited_by_count': 605
'year': 2019, 'works_count': 5, 'cited_by_count': 626
'year': 2018, 'works_count': 7, 'cited_by_count': 562
'year': 2017, 'works_count': 3, 'cited_by_count': 519
'year': 2016, 'works_count': 8, 'cited_by_count': 629
'year': 2015, 'works_count': 0, 'cited_by_count': 662
'year': 2014, 'works_count': 8, 'cited_by_count': 851
'year': 2013, 'works_count': 8, 'cited_by_count': 859
'year': 2012, 'works_count': 1, 'cited_by_count': 812


In [4]:
#A.columns
#['id', 'orcid', 'twitter_id', 'wikipedia_id', 'name', 'n_works', 'n_cit', 'n_cit_year', 'impact_factor', 'h_index', 'i10_index', 'last_institution_id', 'x_concepts']
#'id', 'name', 'n_works', 'n_cit', 'n_cit_year', 'impact_factor', 'h_index', 'i10_index'

max_year=2022

A = A.apply(lambda row: encoding_n_works_cit(row, max_year), axis=1)
A_attr = A[['id', 'n_works', 'n_cit', 'impact_factor', 'h_index', 'i10_index']]
print(A_attr.shape)
A_attr.to_csv(fname_base+'attributes\\authors_numeric.csv', index=False)

A_text = A[['id', 'name']]
print(A_text.shape)
A_text.to_csv(fname_base+'attributes\\authors_text.csv', index=False)



(15401, 6)
(15401, 2)


PAPERS

In [5]:
P = pd.read_csv(fname_base+'papers.csv')
P.columns

Index(['id', 'doi', 'title', 'type', 'version', 'publication_date',
       'publication_year', 'num_citations', 'num_citations_per_year'],
      dtype='object')

In [8]:
P_conc = pd.read_csv(fname_base+'paper_concepts.csv')
papers = P_conc['paper_id'].drop_duplicates().tolist()
print(len(papers))

10173


In [10]:
t = P['type'].drop_duplicates().tolist()
print(t)

v = P['version'].drop_duplicates().tolist()
print(v)

y = P['publication_year'].drop_duplicates().tolist()
print(y)


['book']
['NONE']
[2019, 2020, 2021, 2022]


In [9]:
P_all = pd.read_csv('D:\\PhD\\Openalex\\papers.csv')
t = P_all['type'].drop_duplicates().tolist()
print(t)

['book', 'journal-article', 'proceedings-article']


In [25]:
#n_works_by_year, n_cit_by_year, 

def encoding_n_cit_papers(row, max_year): 
    n_cit = row['num_citations']
    if n_cit>0:
        try:
            data_years = row['num_citations_per_year'][2:-2].split('}, {')
            for data_y in data_years:
                d = data_y.split(', ')
                year = int(d[0].split(': ')[1])
                if year > max_year:
                    n_cit -= int(d[1].split(': ')[1])
            row['num_citations'] = n_cit
        except:
            pass
    return row

In [27]:
#TEST

row = P.head(1)

n_cit = row['num_citations']
print('N cit', n_cit)
data_years = row['num_citations_per_year'].tolist()[0]
data_years = data_years[2:-2]
data_y = data_years.split('}, {')
for d in data_y:
    print(d)

N cit 0    1786
Name: num_citations, dtype: int64
'year': 2023, 'cited_by_count': 12
'year': 2022, 'cited_by_count': 23
'year': 2021, 'cited_by_count': 65
'year': 2020, 'cited_by_count': 65
'year': 2019, 'cited_by_count': 55
'year': 2018, 'cited_by_count': 35
'year': 2017, 'cited_by_count': 49
'year': 2016, 'cited_by_count': 80
'year': 2015, 'cited_by_count': 97
'year': 2014, 'cited_by_count': 110
'year': 2013, 'cited_by_count': 131
'year': 2012, 'cited_by_count': 97


In [26]:
max_year=2022
#Index(['id', 'doi', 'title', 'type', 'version', 'publication_date', 'publication_year', 'num_citations', 'num_citations_per_year'],

P = P.apply(lambda row: encoding_n_cit_papers(row, max_year), axis=1)
P_attr = P[['id', 'publication_year', 'num_citations']]
print(P_attr.shape)
P_attr.to_csv(fname_base+'attributes\\papers_numeric.csv', index=False)

P_text = P[['id', 'title']]
print(P_text.shape)
P_text.to_csv(fname_base+'attributes\\papers_text.csv', index=False)



(10174, 3)
(10174, 2)


In [2]:
#abstracts
P_abs =  pd.read_csv(fname_base+'paper_abstracts.csv', delimiter=';')
P_abs_ok = P_abs[P_abs['abstract'] != 'NONE']
P_abs_ok.shape

(4519, 2)

In [35]:
#count max number of tokens

import nltk
from nltk.tokenize import RegexpTokenizer


en = "i ccc!bcc the a of the abc...ccc dd on aaa 28 abc 19 "

print(len(nltk.word_tokenize(en)))

tokenizer = RegexpTokenizer(r'\w+')
print(len(tokenizer.tokenize(en)))
for t in tokenizer.tokenize(en):
    print(t)

17
15
i
ccc
bcc
the
a
of
the
abc
ccc
dd
on
aaa
28
abc
19


In [52]:
import nltk
#from nltk.tokenize import RegexpTokenizer

def count_tokens(text):
    #tokenizer = RegexpTokenizer(r'\w+')
    #return len(tokenizer.tokenize(text))
    return len(nltk.word_tokenize(text))
    

In [57]:
P_abs_ok['count_tokens'] = P_abs_ok['abstract'].apply(count_tokens)
print('Max number of tokens', max(P_abs_ok['count_tokens'].tolist()))

Max number of tokens 1070


C:\Users\Liliana\AppData\Local\Temp\ipykernel_9564\1714201464.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_abs_ok['count_tokens'] = P_abs_ok['abstract'].apply(count_tokens)


In [53]:
#prendiamo gli abstract. Se l'abstract c'è, lo splittiamo in max 5 parti (256x4=1024<1070). Ogni parte 
# viene passata a SBERT, e poi viene calcolata la media


def processing_abstract(abstract): #return abstract into list with element of max 256 tokens
    if abstract == 'NONE':
        return []
    #tokenizer = RegexpTokenizer(r'\w+')
    #tokens = tokenizer.tokenize(abstract)
    tokens = tokenizer.tokenize(abstract)
    max_tk = 2 #256
    l = len(tokens)//max_tk+1
    print(l)
    lista = []
    for i in range (l):
        lista[i] = tokens[i*max_tk: i*max_tk+l]
        print(lista[i])




In [54]:
processing_abstract('Ciao bella, come stai? Io bene!')

NameError: name 'tokenizer' is not defined

In [43]:
import nltk
from nltk.tokenize import RegexpTokenizer

text = 'Ciao bella come stai?' #['Ciao bella', 'come stai?']
tokenizer = RegexpTokenizer(r'\w+')
tokens = tokenizer.tokenize(text)
ret = []
for i in range(2):
    start = i*2
    end = start + 2
    first_index = text
    print(tokens[start:end])



['Ciao', 'bella']
['come', 'stai']


In [53]:
text = 'Ciao bella come stai? Io bene!'
#tokenizer = RegexpTokenizer(r'\w+')
#tokens = tokenizer.tokenize(text)
tokens = nltk.word_tokenize(text)
print(len(tokens))
print(tokens)

8
['Ciao', 'bella', 'come', 'stai', '?', 'Io', 'bene', '!']


In [52]:
from nltk.tokenize.treebank import TreebankWordDetokenizer
TreebankWordDetokenizer().detokenize(tokens)

'Ciao bella come stai Io bene'

In [8]:
import nltk
from nltk.tokenize.treebank import TreebankWordDetokenizer

def long_text_splitting(abstract):
    if abstract == 'NONE':
        return ['']
    max_tk = 256 #3
    tokens = nltk.word_tokenize(abstract)
    l = len(tokens)//max_tk+1
    lista = ['']*l
    for i in range(l):
        tk_sub = tokens[i*max_tk: max(i*max_tk+l-1, len(tokens))]
        lista[i] = TreebankWordDetokenizer().detokenize(tk_sub)
    return lista


In [67]:
#TEST

abstract = 'Ciao bella! Come stai? Spero vada tutto bene...'
print(long_text_splitting(abstract))
print(long_text_splitting('NONE'))

['Ciao bella!', 'Come stai?', 'Spero vada tutto', 'bene ...']
['']


In [16]:
P_abs = pd.read_csv(fname_base+'paper_abstracts.csv', delimiter=';')
print(P_abs['id'].drop_duplicates().shape)
P_abs['abstract'] = P_abs['abstract'].apply(long_text_splitting)
print(P_abs.shape)
P_abs = P_abs.explode('abstract')
print(P_abs.shape)
print(P_abs['id'].drop_duplicates().shape)
#P_abs.to_csv(fname_base+'\\attributes\\paper_abstracts_text_splitted.csv', index=False)
P_abs.head(5)
P_abs = P_abs.groupby('id').agg({'abstract': lambda x: list(x)}).reset_index()
print(P_abs.shape)

(10173,)
(10174, 2)
(10919, 2)
(10173,)
(10173, 2)


In [12]:
#check lista di lunghezza massima
test = P_abs['abstract'].tolist()
l_max = 0
a_max = []
for lx in test:
    if len(lx) > l_max:
        l_max = len(lx)
        a_max = lx
print(l_max)
for x in a_max:
    print(x)

5
These Guidelines are one of the results of the four-year research project “ Open Online Learning for Digital and Networked Society ” (2017-2021). The project objective was to enable university teachers to design open and online learning through open and online learning curriculum and environment applying learning analytics as a metacognitive tool and creating open and online learning assessment and recognition practices, responding to the needs of digital and networked society . The research of the project resulted in 10 scientific publications and 2 studies prepared by Vytautas Magnus university Institute of Innovative Studies research team in collaboration with their international research partners from Germany, Spain and Portugal . The final stage of the research attempted creating open and online learning assessment and recognition practices, responding to the learner needs in contemporary digital and networked society . The need for open learning recognition has been increasing 

In [9]:
data = pd.read_csv(fname_base+'\\attributes\\paper_abstracts_text_splitted.csv')
data.head(5)
abstracts = data['abstract'].tolist()
print(len(abstracts))
max_len = 0
ax = []
for a in abstracts:
    if len(a) > max_len:
        max_len = len(a)
        ax = a
print(max_len)
print(ax)
print(len(nltk.word_tokenize(ax)))


10174
18765
['These Guidelines are one of the results of the four-year research project “ Open Online Learning for Digital and Networked Society ” (2017-2021). The project objective was to enable university teachers to design open and online learning through open and online learning curriculum and environment applying learning analytics as a metacognitive tool and creating open and online learning assessment and recognition practices, responding to the needs of digital and networked society . The research of the project resulted in 10 scientific publications and 2 studies prepared by Vytautas Magnus university Institute of Innovative Studies research team in collaboration with their international research partners from Germany, Spain and Portugal . The final stage of the research attempted creating open and online learning assessment and recognition practices, responding to the learner needs in contemporary digital and networked society . The need for open learning recognition has been

In [4]:
P_conc = pd.read_csv(fname_base+'paper_concepts.csv')
P_conc.head(3)

,paper_id,concept_id,concept_name,level
0,W1527571715,C115286129,Cybernetics,2
1,W1527571715,C41008148,Computer science,0
2,W1527571715,C2775924081,Control (management),2


In [8]:
concepts = pd.read_csv('D:\\PhD\\Openalex\\OpenAlex concepts in use (17 August 2022) - concepts.csv')
concepts.tail(3)

,openalex_id,display_name,normalized_name,level,wikidata_id,parent_display_names,parent_ids
65023,https://openalex.org/c2909947724,Subcutaneous port,subcutaneous port,5,https://www.wikidata.org/wiki/Q552258,"Embolic stroke, Gamma knife, Gastroesophageal ...","https://openalex.org/C2910003425, https://open..."
65024,https://openalex.org/c2910733956,Modified Radical Mastectomy,modified radical mastectomy,5,https://www.wikidata.org/wiki/Q848318,Mastectomy,https://openalex.org/C2777757722
65025,https://openalex.org/c3019165603,Antiretroviral treatment,antiretroviral treatment,5,https://www.wikidata.org/wiki/Q583050,Antiretroviral therapy,https://openalex.org/C2993143319


In [5]:
labels = pd.read_csv(fname_base+'A_labels.csv')['first_concept_name'].drop_duplicates().tolist()
labels

['Engineering',
 'Philosophy and Art',
 'Psychology, Sociology and History',
 'Mathematics',
 'Economics and Business',
 'Natural/Earth science and Medicine',
 'Physics',
 'Political science']

In [ ]:
#lista di dizionari. Le pos in lista corrispondono a lev mappato (chiave)

In [6]:
#concept_id pronto per confronto con nostro id
def preprocessing_concepts(df):
    df['openalex_id'] = df['openalex_id'].apply(clean_id)
    df['parent_ids'] = df['parent_ids'].apply(clean_ids)
    return df

def clean_id(id):
    return id.replace('https://openalex.org/', '').upper()

def clean_ids(ids):
    try:
        lids = ids.split(', ')
        return [clean_id(id) for id in lids]
    except:
        return []
    
'''['Engineering',
 'Philosophy and Art',
 'Psychology, Sociology and History',
 'Mathematics',
 'Economics and Business',
 'Natural/Earth science and Medicine',
 'Physics',
 'Political science']'''

def mapping_label(c0):
    if c0 == 'C17744445': return 'Political science'
    if c0 == 'C138885662' or c0 == 'C142362112': return 'Philosophy and Art'
    if c0 == 'C162324750' or c0 == 'C144133560': return 'Economics and Business'
    if c0 == 'C15744967' or c0 == 'C144024400' or c0 == 'C95457728': return 'Psychology, Sociology and History'
    if c0 == 'C33923547': return 'Mathematics'
    if c0 == 'C71924100' or c0 == 'C86803240' or c0 == 'C127313418' or c0 == 'C185592680' or c0 == 'C205649164' or c0 == 'C192562407' or c0 == 'C39432304': 
        return 'Natural/Earth science and Medicine'
    if c0 == 'C127413603': return 'Engineering'
    if c0 == 'C121332964': return 'Physics'
    return 'Computer science'
    
    

def mapping_labels(df):
    df_lev0 = df[df['level'] == 0]
    df_lev0['label'] = df_lev0['openalex_id'].apply(mapping_label)
    return df_lev0.set_index('openalex_id')['label'].to_dict()


#dizionario mapping lev X --> lev X-1
def mapping_lev(df, lev):
    df_lev = df[df['level'] == lev]
    return df_lev.set_index('openalex_id')['parent_ids'].to_dict()



In [76]:
#TEST

concepts = preprocessing_concepts(concepts)
test = mapping_lev(concepts.tail(3), 5)
test

{'C2909947724': ['C2910003425',
  'C2994042627',
  'C2909600360',
  'C3017561025',
  'C2993530982',
  'C2992393070'],
 'C2910733956': ['C2777757722'],
 'C3019165603': ['C2993143319']}

In [85]:
for i in range(1, 6)[::-1]:
    print(i)

5
4
3
2
1


In [9]:
concepts = preprocessing_concepts(concepts)

mappings = [{}]*6 
mappings[0] = mapping_labels(concepts)

for i in range(1, 6)[::-1]:
    mappings[i] = mapping_lev(df=concepts, lev=i)

C:\Users\Liliana\AppData\Local\Temp\ipykernel_10272\1936602737.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lev0['label'] = df_lev0['openalex_id'].apply(mapping_label)


In [37]:
#TODO: se più di 1 parent non lo so fare ricorsivo. 

def get_label(row, mapp):
    level = row['level']
    value = row['concept_id']
    lab = get_labels(level, value, mapp) #is a list with duplicates
    row['label'] = list(set(lab))
    return row

def get_labels(lev, conc, mapp):
    lab = []  
    i = lev
    conc_cor = conc
    while i>0:
        #conc_sup = [mapp[lev][c] for c in conc]
        conc_sup = []
        for c in conc_cor:
            c_sup = mapp[i][c]
            for cx in c_sup:
                conc_sup.append(cx)
        conc_cor = conc_sup
        i -= 1
    #i==0
    for c in conc_cor:
        lab.append(mapp[0][c]) #prendi label corrispondente
    return lab

'''
def get_label(level, concept, mapp):
    value = mapp[level][concept] #è una lista 
    if level == 0: 
        return value
    return get_label(level-1, value, mapp)
'''

def aggr_concepts(concepts):
    ret = []
    for c in concepts:
        ret.extend(c)
    return list(set(ret))



In [34]:
#TEST

data = pd.read_csv(fname_base+'paper_concepts.csv').tail(7)
data['concept_id'] = data['concept_id'].apply(lambda x: [x])
print(data)
data = data.apply(lambda row: get_label(row, mappings), axis=1)
print('after')
data = data[['paper_id', 'label']]
print(data)

#data = data.groupby('paper_id').agg({'label': lambda x: ' '.join(x)})
data = data.groupby('paper_id').agg({'label': lambda x: aggr_concepts(x)})
print('finally')
print(data)

           paper_id    concept_id               concept_name  level
115126  W4368319456   [C38652104]          Computer security      1
115127  W4368319456  [C166957645]                Archaeology      1
115128  W4368319456   [C95457728]                    History      0
115129   W977502370   [C41008148]           Computer science      0
115130   W977502370  [C140073362]             Soft computing      3
115131   W977502370  [C154945302]    Artificial intelligence      1
115132   W977502370   [C50644808]  Artificial neural network      2
after
           paper_id                                              label
115126  W4368319456                                 [Computer science]
115127  W4368319456  [Natural/Earth science and Medicine, Psycholog...
115128  W4368319456                [Psychology, Sociology and History]
115129   W977502370                                 [Computer science]
115130   W977502370             [Computer science, Philosophy and Art]
115131   W977502370     

In [38]:
P_conc = pd.read_csv(fname_base+'paper_concepts.csv')
P_conc['label'] = 'NONE'

P_conc['concept_id'] = P_conc['concept_id'].apply(lambda x: [x])
P_conc = P_conc.apply(lambda row: get_label(row, mappings), axis=1)
P_conc_lab = P_conc[['paper_id', 'label']]

P_conc_lab = P_conc_lab.groupby('paper_id').agg({'label': lambda x: aggr_concepts(x)}).reset_index()
P_conc_lab.to_csv(fname_base+'attributes\\paper_concepts_lab.csv', index=False)
P_conc_lab.head(5)

,paper_id,label
0,W10124311,[Computer science]
1,W1156742600,"[Engineering, Computer science, Economics and ..."
2,W120464988,"[Natural/Earth science and Medicine, Mathemati..."
3,W126508979,"[Engineering, Political science, Computer scie..."
4,W127766032,"[Computer science, Philosophy and Art]"


In [39]:
P_conc_lab.head(5)

,paper_id,label
0,W10124311,[Computer science]
1,W1156742600,"[Engineering, Computer science, Economics and ..."
2,W120464988,"[Natural/Earth science and Medicine, Mathemati..."
3,W126508979,"[Engineering, Political science, Computer scie..."
4,W127766032,"[Computer science, Philosophy and Art]"


In [48]:
P_ids = P_conc_lab['paper_id'].tolist()

P_conc_lab_ohe = P_conc_lab['label'].str.join('|').str.get_dummies()
P_conc_lab_ohe['paper_id'] = P_ids

# shift column 'paper_id' to first position 
first_column = P_conc_lab_ohe.pop('paper_id') 
# insert column  
P_conc_lab_ohe.insert(0, 'paper_id', first_column) 
P_conc_lab_ohe.to_csv(fname_base+'attributes\\paper_concepts_lab_ohe.csv', index=False)
P_conc_lab_ohe.columns

Index(['paper_id', 'Computer science', 'Economics and Business', 'Engineering',
       'Mathematics', 'Natural/Earth science and Medicine',
       'Philosophy and Art', 'Physics', 'Political science',
       'Psychology, Sociology and History'],
      dtype='object')

In [49]:
P_conc_lab_ohe.head(5)

,paper_id,Computer science,Economics and Business,Engineering,Mathematics,Natural/Earth science and Medicine,Philosophy and Art,Physics,Political science,"Psychology, Sociology and History"
0,W10124311,1,0,0,0,0,0,0,0,0
1,W1156742600,1,1,1,0,0,0,0,0,0
2,W120464988,1,0,0,1,1,0,1,0,0
3,W126508979,1,0,1,0,0,1,0,1,0
4,W127766032,1,0,0,0,0,1,0,0,0
